# SECTION 1 — EXPLORATORY DATA ANALYSIS

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from IPython.display import display

import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# 1) Initial inspection
df = pd.read_csv("Loan_default.csv")  # Replace with your actual data source
display(df.head())

print(f"Shape: {df.shape}")
df.info()

numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
if numeric_cols:
    display(df[numeric_cols].describe().T)
else:
    print("No numeric features found.")

cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
if cat_cols:
    cat_summary = []
    for col in cat_cols:
        vc = df[col].value_counts(dropna=False)
        cat_summary.append(
            {
                "column": col,
                "dtype": str(df[col].dtype),
                "n_unique": df[col].nunique(dropna=False),
                "top_value": vc.index[0] if len(vc) > 0 else np.nan,
                "top_pct": vc.iloc[0] / len(df) if len(vc) > 0 else np.nan,
                "missing": int(df[col].isna().sum()),
            }
        )
    display(pd.DataFrame(cat_summary).set_index("column"))
else:
    print("No categorical features found.")

print(
    f"Concise interpretation: the dataset has {df.shape[0]} rows and {df.shape[1]} columns "
    f"with {len(numeric_cols)} numeric and {len(cat_cols)} categorical-like features. "
    "The next steps are to assess data quality, target balance, and feature distributions."
)

# SECTION 2 — FEATURE ENGINEERING

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Features (all numeric + categorical, excluding target) and target
target_col = "Default"
num_features = [c for c in numeric_cols if c != target_col]
cat_features = [c for c in cat_cols if c != target_col]
feature_cols = num_features + cat_features

X = df[feature_cols].copy()
y = df[target_col].astype(int)

# 1) 80% development, 20% final test (untouched)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 2) Development -> 80% train, 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev,
    test_size=0.20,
    random_state=42,
    stratify=y_dev
)

# Display shapes and default rates
split_summary = pd.DataFrame(
    [
        {"split": "train", "X_shape": X_train.shape, "y_shape": y_train.shape, "default_rate": y_train.mean()},
        {"split": "validation", "X_shape": X_val.shape, "y_shape": y_val.shape, "default_rate": y_val.mean()},
        {"split": "test", "X_shape": X_test.shape, "y_shape": y_test.shape, "default_rate": y_test.mean()},
    ]
)
display(split_summary)

# Preprocessing learned ONLY from training data
numeric_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, num_features),
        ("cat", categorical_preprocess, cat_features),
    ]
)

# Fit on training only (learn imputation stats, scaling stats, and category vocabularies)
preprocessor.fit(X_train)

# Transform train/validation for modeling/tuning
X_train_prepared = preprocessor.transform(X_train)
X_val_prepared = preprocessor.transform(X_val)

print("Prepared train shape:", X_train_prepared.shape)
print("Prepared validation shape:", X_val_prepared.shape)
print("Test split kept untouched until final model configuration is frozen.")

In [ ]:
def create_engineered_features(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()

    # ---- Numeric engineered features (deterministic, no target/stat leakage) ----
    income = out["Income"].astype(float)
    loan_amount = out["LoanAmount"].astype(float)
    months_employed = out["MonthsEmployed"].astype(float)
    age = out["Age"].astype(float)
    num_credit_lines = out["NumCreditLines"].astype(float)
    credit_score = out["CreditScore"].astype(float)
    dti = out["DTIRatio"].astype(float)
    interest_rate_annual_pct = out["InterestRate"].astype(float)
    loan_term_months = out["LoanTerm"].astype(float)

    # Safe denominators
    age_months_available = np.maximum((age - 18.0) * 12.0, 1.0)
    credit_lines_safe = np.maximum(num_credit_lines, 1.0)
    credit_score_safe = np.maximum(credit_score, 1.0)
    loan_term_safe = np.maximum(loan_term_months, 1.0)

    out["LoanToIncome"] = np.where(income != 0, loan_amount / income, 0.0)
    out["EmploymentYears"] = months_employed / 12.0
    out["EmploymentToAgeRatio"] = months_employed / age_months_available
    out["IncomePerCreditLine"] = income / credit_lines_safe
    out["LoanPerCreditLine"] = loan_amount / credit_lines_safe
    out["LoanToCreditScore"] = loan_amount / credit_score_safe
    out["DebtIncomeProxy"] = income * dti

    # Assumptions: Income annual, InterestRate annual %, LoanTerm in months
    out["MonthlyIncome"] = income / 12.0
    out["MonthlyInterestRate"] = interest_rate_annual_pct / 100.0 / 12.0

    r = out["MonthlyInterestRate"].to_numpy(dtype=float)
    n = loan_term_safe.to_numpy(dtype=float)
    p = loan_amount.to_numpy(dtype=float)

    # Amortized payment; if r == 0 then payment = principal / term
    with np.errstate(divide="ignore", invalid="ignore"):
        amort_payment = p * (r * (1.0 + r) ** n) / (((1.0 + r) ** n) - 1.0)
    zero_rate_payment = p / n
    estimated_payment = np.where(r > 0, amort_payment, zero_rate_payment)
    estimated_payment = np.nan_to_num(estimated_payment, nan=0.0, posinf=0.0, neginf=0.0)

    out["EstimatedMonthlyPayment"] = estimated_payment
    out["PaymentToMonthlyIncome"] = np.where(out["MonthlyIncome"] > 0, out["EstimatedMonthlyPayment"] / out["MonthlyIncome"], 0.0)
    out["TotalInterest"] = out["EstimatedMonthlyPayment"] * loan_term_safe - loan_amount

    # ---- Fixed, readable buckets ----
    out["AgeBand"] = pd.cut(
        age,
        bins=[0, 25, 35, 45, 55, 65, np.inf],
        labels=["18-25", "26-35", "36-45", "46-55", "56-65", "66+"],
        include_lowest=True
    )

    out["CreditScoreBand"] = pd.cut(
        credit_score,
        bins=[0, 580, 670, 740, 800, np.inf],
        labels=["Poor", "Fair", "Good", "Very Good", "Excellent"],
        include_lowest=True
    )

    out["EmploymentDurationBand"] = pd.cut(
        out["EmploymentYears"],
        bins=[0, 1, 3, 5, 10, np.inf],
        labels=["0-1y", "1-3y", "3-5y", "5-10y", "10y+"],
        include_lowest=True
    )

    out["InterestRateBand"] = pd.cut(
        interest_rate_annual_pct,
        bins=[0, 5, 10, 15, 20, np.inf],
        labels=["0-5%", "5-10%", "10-15%", "15-20%", "20%+"],
        include_lowest=True
    )

    out["DTIRatioBand"] = pd.cut(
        dti,
        bins=[0, 0.20, 0.36, 0.43, 0.50, np.inf],
        labels=["<=20%", "20-36%", "36-43%", "43-50%", "50%+"],
        include_lowest=True
    )

    out["LoanTermBand"] = pd.cut(
        loan_term_months,
        bins=[0, 12, 24, 36, 48, 60, np.inf],
        labels=["<=12m", "13-24m", "25-36m", "37-48m", "49-60m", "60m+"],
        include_lowest=True
    )

    out["LoanToIncomeBand"] = pd.cut(
        out["LoanToIncome"],
        bins=[0, 0.5, 1.0, 2.0, 3.0, np.inf],
        labels=["<=0.5x", "0.5-1x", "1-2x", "2-3x", "3x+"],
        include_lowest=True
    )

    # ---- Only requested categorical interactions ----
    edu = out["Education"].astype("string").fillna("Unknown")
    emp = out["EmploymentType"].astype("string").fillna("Unknown")
    mar = out["MaritalStatus"].astype("string").fillna("Unknown")
    dep = out["HasDependents"].astype("string").fillna("Unknown")
    mort = out["HasMortgage"].astype("string").fillna("Unknown")
    purpose = out["LoanPurpose"].astype("string").fillna("Unknown")
    cosigner = out["HasCoSigner"].astype("string").fillna("Unknown")
    cs_band = out["CreditScoreBand"].astype("string").fillna("Unknown")

    out["Education_EmploymentType"] = edu + "__" + emp
    out["MaritalStatus_HasDependents"] = mar + "__" + dep
    out["HasMortgage_LoanPurpose"] = mort + "__" + purpose
    out["HasCoSigner_CreditScoreBand"] = cosigner + "__" + cs_band

    return out


# Apply once; keep original df unchanged
df_features = create_engineered_features(df)

display(
    df_features[
        [
            "LoanToIncome", "EmploymentYears", "EmploymentToAgeRatio", "IncomePerCreditLine",
            "LoanPerCreditLine", "LoanToCreditScore", "DebtIncomeProxy", "MonthlyIncome",
            "MonthlyInterestRate", "EstimatedMonthlyPayment", "PaymentToMonthlyIncome",
            "TotalInterest", "AgeBand", "CreditScoreBand", "EmploymentDurationBand",
            "InterestRateBand", "DTIRatioBand", "LoanTermBand", "LoanToIncomeBand",
            "Education_EmploymentType", "MaritalStatus_HasDependents",
            "HasMortgage_LoanPurpose", "HasCoSigner_CreditScoreBand"
        ]
    ].head()
)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# ── 1. Define feature sets (exclude LoanID and Default) ──────────────────────
ENG_NUM_FEATURES = [
    "Age", "Income", "LoanAmount", "CreditScore", "MonthsEmployed",
    "NumCreditLines", "InterestRate", "LoanTerm", "DTIRatio",
    "LoanToIncome", "EmploymentYears", "EmploymentToAgeRatio",
    "IncomePerCreditLine", "LoanPerCreditLine", "LoanToCreditScore",
    "DebtIncomeProxy", "MonthlyIncome", "MonthlyInterestRate",
    "EstimatedMonthlyPayment", "PaymentToMonthlyIncome", "TotalInterest",
]

ENG_CAT_FEATURES = [
    "Education", "EmploymentType", "MaritalStatus", "HasMortgage",
    "HasDependents", "LoanPurpose", "HasCoSigner",
    "AgeBand", "CreditScoreBand", "EmploymentDurationBand",
    "InterestRateBand", "DTIRatioBand", "LoanTermBand", "LoanToIncomeBand",
    "Education_EmploymentType", "MaritalStatus_HasDependents",
    "HasMortgage_LoanPurpose", "HasCoSigner_CreditScoreBand",
]

target_col = "Default"

# ── 2. Apply feature engineering & split (no leakage) ────────────────────────
df_features = create_engineered_features(df)

# Cast band/categorical cols to str; fill nulls with __MISSING__
for c in ENG_CAT_FEATURES:
    df_features[c] = df_features[c].astype(str).fillna("__MISSING__").replace("nan", "__MISSING__")

X_all = df_features[ENG_NUM_FEATURES + ENG_CAT_FEATURES].copy()
y_all = df_features[target_col].astype(int)


X_dev2, X_test2, y_dev2, y_test2 = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42, stratify=y_all
)
X_train2, X_val2, y_train2, y_val2 = train_test_split(
    X_dev2, y_dev2, test_size=0.20, random_state=42, stratify=y_dev2
)

print("Train:", X_train2.shape, "  Val:", X_val2.shape, "  Test:", X_test2.shape)

# ── 3. Numeric imputation (median from train only) ────────────────────────────

num_imputer = SimpleImputer(strategy="median")
num_imputer.fit(X_train2[ENG_NUM_FEATURES])

X_train_num = num_imputer.transform(X_train2[ENG_NUM_FEATURES]).astype("float32")
X_val_num   = num_imputer.transform(X_val2[ENG_NUM_FEATURES]).astype("float32")
X_test_num  = num_imputer.transform(X_test2[ENG_NUM_FEATURES]).astype("float32")

# ── 4. Keras Normalization layer (adapted on train only) ──────────────────────
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(X_train_num)
print("Normalization mean shape:", normalizer.mean.shape)

# ── 5. StringLookup layers per categorical feature (fit on train only) ────────
string_lookup_layers = {}
for feat in ENG_CAT_FEATURES:
    train_vocab = X_train2[feat].astype(str).values
    lookup = layers.StringLookup(
        output_mode="int",
        mask_token=None,
        oov_token="[UNK]"
    )
    lookup.adapt(train_vocab)
    string_lookup_layers[feat] = lookup
    print(f"  {feat}: vocab_size={lookup.vocabulary_size()}")

# ── 6. Helper: build dict of arrays for tf.data ──────────────────────────────
def build_feature_dict(X_num_arr, X_cat_df):
    feat_dict = {"numeric": X_num_arr}
    for feat in ENG_CAT_FEATURES:
        feat_dict[feat] = X_cat_df[feat].astype(str).values
    return feat_dict

train_feat = build_feature_dict(X_train_num, X_train2)
val_feat   = build_feature_dict(X_val_num,   X_val2)
test_feat  = build_feature_dict(X_test_num,  X_test2)

# ── 7. tf.data datasets ───────────────────────────────────────────────────────
BATCH_SIZE   = 2048
SHUFFLE_SEED = 42

def make_dataset(feat_dict, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((feat_dict, labels.values.astype("float32")))
    if shuffle:
        ds = ds.shuffle(buffer_size=10_000, seed=SHUFFLE_SEED)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_feat, y_train2, shuffle=True)
val_ds   = make_dataset(val_feat,   y_val2,   shuffle=False)
test_ds  = make_dataset(test_feat,  y_test2,  shuffle=False)

print("train_ds:", train_ds)
print("val_ds:  ", val_ds)
print("test_ds: ", test_ds)

# ── 8. Sanity checks ──────────────────────────────────────────────────────────
sample_batch, sample_labels = next(iter(train_ds))
print("\nNumeric batch shape :", sample_batch["numeric"].shape)
for feat in ENG_CAT_FEATURES[:3]:
    print(f"Cat '{feat}' batch shape:", sample_batch[feat].shape)

# Verify no inf / nan in numeric arrays
for name, arr in [("train", X_train_num), ("val", X_val_num), ("test", X_test_num)]:
    assert np.all(np.isfinite(arr)), f"Non-finite values found in {name} numeric array!"
print("\nAll numeric arrays are finite. Preprocessing complete.")

# SECTION 3 — NEURAL NETWORK DEVELOPMENT

In [ ]:
import random
import time
from sklearn.metrics import average_precision_score

# 1) Reproducibility / environment
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    determinism_status = "enabled"
except Exception as e:
    determinism_status = f"not fully enabled ({e})"

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)
print("Deterministic ops:", determinism_status)

# 2) Rebuild datasets with requested batch size
BASELINE_BATCH_SIZE = 512

def make_dataset_baseline(feat_dict, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((feat_dict, labels.values.astype("float32")))
    if shuffle:
        ds = ds.shuffle(buffer_size=10_000, seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BASELINE_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds_baseline = make_dataset_baseline(train_feat, y_train2, shuffle=True)
val_ds_baseline = make_dataset_baseline(val_feat, y_val2, shuffle=False)
test_ds_baseline = make_dataset_baseline(test_feat, y_test2, shuffle=False)

# 3) Simple neural baseline
tf.keras.backend.clear_session()

numeric_input = keras.Input(shape=(len(ENG_NUM_FEATURES),), name="numeric", dtype=tf.float32)
numeric_branch = normalizer(numeric_input)

categorical_inputs = []
categorical_branches = []

for feat in ENG_CAT_FEATURES:
    inp = keras.Input(shape=(), name=feat, dtype=tf.string)
    categorical_inputs.append(inp)

    idx = string_lookup_layers[feat](inp)
    one_hot = layers.CategoryEncoding(
        num_tokens=string_lookup_layers[feat].vocabulary_size(),
        output_mode="one_hot"
    )(idx)
    categorical_branches.append(one_hot)

all_features = layers.Concatenate()([numeric_branch] + categorical_branches)
x = layers.Dense(128, activation="relu")(all_features)
x = layers.Dropout(0.20)(x)
output = layers.Dense(1, activation="sigmoid", dtype="float32")(x)

simple_baseline = keras.Model(inputs=[numeric_input] + categorical_inputs, outputs=output, name="simple_baseline")

simple_baseline.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.AUC(curve="PR", name="pr_auc"),
        keras.metrics.AUC(curve="ROC", name="roc_auc"),
    ],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_pr_auc",
        mode="max",
        patience=10,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_pr_auc",
        mode="max",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.TerminateOnNaN(),
]

start_time = time.perf_counter()

history = simple_baseline.fit(
    train_ds_baseline,
    validation_data=val_ds_baseline,
    epochs=100,
    callbacks=callbacks,
    verbose=2,
)

training_time_sec = time.perf_counter() - start_time

# 4) Final validation / test probabilities and PR-AUC from stored probabilities
baseline_val_proba = simple_baseline.predict(val_ds_baseline, verbose=0).ravel()
baseline_test_proba = simple_baseline.predict(test_ds_baseline, verbose=0).ravel()

baseline_val_pr_auc = average_precision_score(y_val2, baseline_val_proba)
baseline_test_pr_auc = average_precision_score(y_test2, baseline_test_proba)

# 5) Training summary
best_idx = int(np.argmax(history.history["val_pr_auc"]))
best_epoch = best_idx + 1
best_val_pr_auc_keras = float(history.history["val_pr_auc"][best_idx])
best_train_pr_auc = float(history.history["pr_auc"][best_idx])
param_count = simple_baseline.count_params()

default_rate = float(y_train2.mean())
gap = best_train_pr_auc - best_val_pr_auc_keras

if gap > 0.03:
    fit_note = f"Possible overfitting: train-val PR-AUC gap = {gap:.4f}."
elif best_train_pr_auc < default_rate * 1.25 and best_val_pr_auc_keras < default_rate * 1.25:
    fit_note = f"Possible underfitting: PR-AUC is close to the default rate ({default_rate:.4f})."
else:
    fit_note = "No strong overfitting/underfitting signal."

results_baseline = pd.DataFrame(
    {
        "model": ["simple_baseline"],
        "params": [param_count],
        "best_epoch": [best_epoch],
        "training_time_sec": [round(training_time_sec, 2)],
        "val_pr_auc_keras": [round(best_val_pr_auc_keras, 6)],
        "val_pr_auc_ap": [round(baseline_val_pr_auc, 6)],
        "test_pr_auc_ap": [round(baseline_test_pr_auc, 6)],
        "train_pr_auc_at_best_epoch": [round(best_train_pr_auc, 6)],
        "default_rate_train": [round(default_rate, 6)],
        "fit_assessment": [fit_note],
    }
)

display(results_baseline)
print(fit_note)
simple_baseline.summary()

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, brier_score_loss, log_loss,
    confusion_matrix
)

# ── 1. Validation probabilities (already computed) ────────────────────────────
val_proba  = baseline_val_proba
test_proba = baseline_test_proba

y_val_arr  = y_val2.values
y_test_arr = y_test2.values

# ── 2. Threshold selection on VALIDATION set only ────────────────────────────

# a) Default threshold
thresh_default = 0.5

# b) F1-maximising threshold
thresholds_grid = np.linspace(0.01, 0.99, 500)
f1_scores = [f1_score(y_val_arr, (val_proba >= t).astype(int), zero_division=0) for t in thresholds_grid]
thresh_f1 = float(thresholds_grid[np.argmax(f1_scores)])

# c) Business threshold: max Precision with Recall >= 60%
business_thresh = thresh_default
best_prec = -1.0
for t in thresholds_grid:
    preds = (val_proba >= t).astype(int)
    rec  = recall_score(y_val_arr, preds, zero_division=0)
    prec = precision_score(y_val_arr, preds, zero_division=0)
    if rec >= 0.60 and prec > best_prec:
        best_prec = prec
        business_thresh = float(t)

print(f"Threshold — Default : {thresh_default:.4f}")
print(f"Threshold — F1-max  : {thresh_f1:.4f}  (val F1={max(f1_scores):.4f})")
print(f"Threshold — Business: {business_thresh:.4f}  (val Prec={best_prec:.4f} @ Recall>=60%)")

# ── 3. Final test evaluation ──────────────────────────────────────────────────

def threshold_metrics(y_true, proba, threshold, label):
    preds = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    return {
        "threshold_label"       : label,
        "threshold"             : round(threshold, 4),
        "precision"             : round(precision_score(y_true, preds, zero_division=0), 4),
        "recall"                : round(recall_score(y_true, preds, zero_division=0), 4),
        "f1"                    : round(f1_score(y_true, preds, zero_division=0), 4),
        "predicted_positive_rate": round(preds.mean(), 4),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
    }

# Threshold-independent metrics (computed once)
roc_auc  = roc_auc_score(y_test_arr, test_proba)
pr_auc   = average_precision_score(y_test_arr, test_proba)
brier    = brier_score_loss(y_test_arr, test_proba)
logloss  = log_loss(y_test_arr, test_proba)

print(f"\nThreshold-independent  |  ROC-AUC={roc_auc:.4f}  PR-AUC={pr_auc:.4f}"
      f"  Brier={brier:.4f}  LogLoss={logloss:.4f}")

# Threshold-dependent rows
rows_thresh = []
for label, thresh in [("default", thresh_default), ("f1_max", thresh_f1), ("business", business_thresh)]:
    row = threshold_metrics(y_test_arr, test_proba, thresh, label)
    row.update({"roc_auc": round(roc_auc, 4), "pr_auc": round(pr_auc, 4),
                "brier_score": round(brier, 4), "log_loss": round(logloss, 4)})
    rows_thresh.append(row)

df_thresh = pd.DataFrame(rows_thresh)
display(df_thresh)

# ── 4. Precision / Recall / Lift @ k% ────────────────────────────────────────
n_test      = len(y_test_arr)
n_defaults  = y_test_arr.sum()
sorted_idx  = np.argsort(test_proba)[::-1]          # descending by score

rows_topk = []
for k in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    n_flagged        = max(1, int(np.ceil(k * n_test)))
    flagged_idx      = sorted_idx[:n_flagged]
    defaults_caught  = int(y_test_arr[flagged_idx].sum())
    precision_k      = defaults_caught / n_flagged
    recall_k         = defaults_caught / n_defaults if n_defaults > 0 else 0.0
    baseline_rate    = n_defaults / n_test
    lift_k           = precision_k / baseline_rate if baseline_rate > 0 else 0.0
    rows_topk.append({
        "k_pct"             : f"{int(k*100)}%",
        "n_flagged"         : n_flagged,
        "defaults_captured" : defaults_caught,
        "precision_at_k"    : round(precision_k, 4),
        "recall_at_k"       : round(recall_k, 4),
        "lift_at_k"         : round(lift_k, 4),
    })

df_topk = pd.DataFrame(rows_topk)
display(df_topk)

# ── 5. Master results DataFrame ───────────────────────────────────────────────
master_records = []
for _, tr in df_thresh.iterrows():
    base = tr.to_dict()
    base["metric_type"] = "threshold_metrics"
    master_records.append(base)

for _, tk in df_topk.iterrows():
    rec = tk.to_dict()
    rec["metric_type"] = "topk_metrics"
    master_records.append(rec)

df_master = pd.DataFrame(master_records)
display(df_master)

In [ ]:
from sklearn.calibration import calibration_curve
import warnings
from sklearn.utils import check_random_state
from sklearn.model_selection import StratifiedShuffleSplit

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    precision_recall_curve, roc_curve, roc_auc_score,
    average_precision_score, brier_score_loss, log_loss,
    confusion_matrix, f1_score, precision_score, recall_score
)
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — VISUALIZATIONS
# ─────────────────────────────────────────────────────────────────────────────

hist = history.history
epochs_ran = range(1, len(hist["loss"]) + 1)

# ── 7.1 Loss & PR-AUC learning curves ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_ran, hist["loss"], label="Train Loss")
axes[0].plot(epochs_ran, hist["val_loss"], label="Val Loss")
axes[0].set_title("Training vs Validation Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Binary Cross-Entropy")
axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs_ran, hist["pr_auc"], label="Train PR-AUC")
axes[1].plot(epochs_ran, hist["val_pr_auc"], label="Val PR-AUC")
axes[1].set_title("Training vs Validation PR-AUC")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("PR-AUC")
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig("learning_curves.png", dpi=150)
plt.show()

# ── 7.2 Precision-Recall curve ────────────────────────────────────────────────
prec_arr, rec_arr, _ = precision_recall_curve(y_test_arr, test_proba)
baseline_rate_test    = y_test_arr.mean()

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(rec_arr, prec_arr, lw=2, label=f"Simple Baseline (PR-AUC={pr_auc:.4f})")
ax.axhline(baseline_rate_test, color="grey", linestyle="--", label=f"No-skill baseline ({baseline_rate_test:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve (Test Set)")
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig("pr_curve.png", dpi=150)
plt.show()

# ── 7.3 ROC curve ─────────────────────────────────────────────────────────────
fpr_arr, tpr_arr, _ = roc_curve(y_test_arr, test_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_arr, tpr_arr, lw=2, label=f"Simple Baseline (ROC-AUC={roc_auc:.4f})")
ax.plot([0, 1], [0, 1], "k--", label="Random")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve (Test Set)")
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=150)
plt.show()

# ── 7.4 Precision, Recall and F1 vs Threshold ────────────────────────────────
thresholds_grid = np.linspace(0.01, 0.99, 500)
prec_t, rec_t, f1_t = [], [], []
for t in thresholds_grid:
    preds = (val_proba >= t).astype(int)
    prec_t.append(precision_score(y_val_arr, preds, zero_division=0))
    rec_t.append(recall_score(y_val_arr, preds, zero_division=0))
    f1_t.append(f1_score(y_val_arr, preds, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds_grid, prec_t, label="Precision")
ax.plot(thresholds_grid, rec_t,  label="Recall")
ax.plot(thresholds_grid, f1_t,   label="F1")
ax.axvline(thresh_f1,       color="green",  linestyle="--", label=f"F1-max ({thresh_f1:.3f})")
ax.axvline(business_thresh, color="orange", linestyle="--", label=f"Business ({business_thresh:.3f})")
ax.axvline(thresh_default,  color="red",    linestyle="--", label=f"Default (0.50)")
ax.set_xlabel("Threshold"); ax.set_ylabel("Score")
ax.set_title("Precision / Recall / F1 vs Decision Threshold (Validation Set)")
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig("threshold_curves.png", dpi=150)
plt.show()

# ── 7.5 Precision@k, Recall@k, Lift@k ────────────────────────────────────────
k_values      = np.linspace(0.01, 0.50, 100)
n_test        = len(y_test_arr)
n_defaults    = y_test_arr.sum()
sorted_idx_t  = np.argsort(test_proba)[::-1]
baseline_rate_v = n_defaults / n_test

prec_k_list, rec_k_list, lift_k_list = [], [], []
for k in k_values:
    n_flagged       = max(1, int(np.ceil(k * n_test)))
    flagged_idx     = sorted_idx_t[:n_flagged]
    defaults_caught = y_test_arr[flagged_idx].sum()
    prec_k          = defaults_caught / n_flagged
    rec_k           = defaults_caught / n_defaults if n_defaults > 0 else 0
    lift_k          = prec_k / baseline_rate_v if baseline_rate_v > 0 else 0
    prec_k_list.append(prec_k); rec_k_list.append(rec_k); lift_k_list.append(lift_k)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, vals, title, ylabel in zip(
    axes,
    [prec_k_list, rec_k_list, lift_k_list],
    ["Precision@k", "Recall@k", "Lift@k"],
    ["Precision", "Recall", "Lift"]
):
    ax.plot(k_values * 100, vals, lw=2)
    if title == "Lift@k":
        ax.axhline(1.0, color="grey", linestyle="--", label="No-skill")
        ax.legend()
    ax.set_xlabel("Top k% of population")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{title} (Test Set)")
    ax.grid(True)
plt.tight_layout()
plt.savefig("topk_curves.png", dpi=150)
plt.show()

# ── 7.6 Confusion matrices ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (label, thresh) in zip(
    axes,
    [("Default (0.50)", thresh_default), (f"F1-max ({thresh_f1:.3f})", thresh_f1), (f"Business ({business_thresh:.3f})", business_thresh)]
):
    cm = confusion_matrix(y_test_arr, (test_proba >= thresh).astype(int))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", ax=ax,
        xticklabels=["Pred No-Default", "Pred Default"],
        yticklabels=["Actual No-Default", "Actual Default"]
    )
    ax.set_title(f"Confusion Matrix\nThreshold: {label}")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()

# ── 7.7 Calibration curve ─────────────────────────────────────────────────────
frac_pos, mean_pred = calibration_curve(y_test_arr, test_proba, n_bins=15, strategy="quantile")

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(mean_pred, frac_pos, marker="o", lw=2, label="Simple Baseline")
ax.plot([0, 1], [0, 1], "k--", label="Perfect Calibration")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.set_title("Calibration Curve (Test Set)")
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig("calibration_curve.png", dpi=150)
plt.show()

# ── 7.8 Score distribution by actual class ───────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(test_proba[y_test_arr == 0], bins=60, alpha=0.6, density=True, label="No Default (0)")
ax.hist(test_proba[y_test_arr == 1], bins=60, alpha=0.6, density=True, label="Default (1)")
ax.axvline(thresh_f1,       color="green",  linestyle="--", label=f"F1-max ({thresh_f1:.3f})")
ax.axvline(business_thresh, color="orange", linestyle="--", label=f"Business ({business_thresh:.3f})")
ax.axvline(thresh_default,  color="red",    linestyle="--", label="Default (0.50)")
ax.set_xlabel("Predicted Probability of Default")
ax.set_ylabel("Density")
ax.set_title("Predicted Score Distribution by Actual Class (Test Set)")
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig("score_distributions.png", dpi=150)
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — PERMUTATION IMPORTANCE
# ─────────────────────────────────────────────────────────────────────────────


N_REPEATS    = 5
MAX_ROWS     = 10_000
PERM_SEED    = 42
rng          = check_random_state(PERM_SEED)

# Stratified subsample from test set
sss = StratifiedShuffleSplit(n_splits=1, test_size=min(MAX_ROWS, len(y_test_arr)) / len(y_test_arr), random_state=PERM_SEED)
sub_idx, _ = next(sss.split(X_test2, y_test2))
X_perm_cat  = X_test2.iloc[sub_idx].copy()
X_perm_num  = X_test_num[sub_idx]
y_perm      = y_test2.iloc[sub_idx].values

# Baseline PR-AUC on subsample
def score_model(X_num_arr, X_cat_df):
    feat_dict = build_feature_dict(X_num_arr, X_cat_df)
    ds = tf.data.Dataset.from_tensor_slices(feat_dict).batch(2048).prefetch(tf.data.AUTOTUNE)
    proba = simple_baseline.predict(ds, verbose=0).ravel()
    return average_precision_score(y_perm, proba)

baseline_perm_score = score_model(X_perm_num, X_perm_cat)
print(f"Baseline PR-AUC on perm-sample: {baseline_perm_score:.4f}")

all_raw_features = ENG_NUM_FEATURES + ENG_CAT_FEATURES
importance_records = []

for feat in all_raw_features:
    decreases = []
    for repeat in range(N_REPEATS):
        if feat in ENG_NUM_FEATURES:
            feat_idx     = ENG_NUM_FEATURES.index(feat)
            X_num_shuf   = X_perm_num.copy()
            shuffled_col = rng.permutation(X_num_shuf[:, feat_idx])
            X_num_shuf[:, feat_idx] = shuffled_col
            shuf_score   = score_model(X_num_shuf, X_perm_cat)
        else:
            X_cat_shuf        = X_perm_cat.copy()
            X_cat_shuf[feat]  = rng.permutation(X_cat_shuf[feat].values)
            shuf_score        = score_model(X_perm_num, X_cat_shuf)
        decreases.append(baseline_perm_score - shuf_score)

    importance_records.append({
        "feature" : feat,
        "mean_decrease_pr_auc": np.mean(decreases),
        "std_decrease_pr_auc" : np.std(decreases),
    })
    print(f"  {feat}: {np.mean(decreases):.5f} ± {np.std(decreases):.5f}")

df_importance = (
    pd.DataFrame(importance_records)
    .sort_values("mean_decrease_pr_auc", ascending=False)
    .reset_index(drop=True)
)
display(df_importance.head(20))

top20 = df_importance.head(20).sort_values("mean_decrease_pr_auc")
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top20["feature"], top20["mean_decrease_pr_auc"], xerr=top20["std_decrease_pr_auc"],
        align="center", alpha=0.8, capsize=3)
ax.set_xlabel("Mean Decrease in PR-AUC")
ax.set_title("Permutation Feature Importance — Top 20 (Test Subsample, 5 Repeats)")
ax.grid(True, axis="x")
plt.tight_layout()
plt.savefig("permutation_importance.png", dpi=150)
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 — INTEREST RATE SENSITIVITY
# ─────────────────────────────────────────────────────────────────────────────

IR_DERIVED = [
    "InterestRate", "MonthlyInterestRate", "EstimatedMonthlyPayment",
    "PaymentToMonthlyIncome", "TotalInterest", "InterestRateBand",
]

ENG_NUM_NO_IR  = [f for f in ENG_NUM_FEATURES if f not in IR_DERIVED]
ENG_CAT_NO_IR  = [f for f in ENG_CAT_FEATURES if f not in IR_DERIVED]

print(f"Booked features  : {len(ENG_NUM_FEATURES)} num + {len(ENG_CAT_FEATURES)} cat")
print(f"Pre-pricing feats: {len(ENG_NUM_NO_IR)} num + {len(ENG_CAT_NO_IR)} cat")

# ── Build pre-pricing datasets ────────────────────────────────────────────────
num_imputer_nir = SimpleImputer(strategy="median")
num_imputer_nir.fit(X_train2[ENG_NUM_NO_IR])

X_train_num_nir = num_imputer_nir.transform(X_train2[ENG_NUM_NO_IR]).astype("float32")
X_val_num_nir   = num_imputer_nir.transform(X_val2[ENG_NUM_NO_IR]).astype("float32")
X_test_num_nir  = num_imputer_nir.transform(X_test2[ENG_NUM_NO_IR]).astype("float32")

normalizer_nir = layers.Normalization(axis=-1)
normalizer_nir.adapt(X_train_num_nir)

string_lookup_nir = {}
for feat in ENG_CAT_NO_IR:
    lookup = layers.StringLookup(output_mode="int", mask_token=None, oov_token="[UNK]")
    lookup.adapt(X_train2[feat].astype(str).values)
    string_lookup_nir[feat] = lookup

def build_feat_dict_nir(X_num_arr, X_cat_df, cat_feats):
    d = {"numeric": X_num_arr}
    for f in cat_feats:
        d[f] = X_cat_df[f].astype(str).values
    return d

train_feat_nir = build_feat_dict_nir(X_train_num_nir, X_train2, ENG_CAT_NO_IR)
val_feat_nir   = build_feat_dict_nir(X_val_num_nir,   X_val2,   ENG_CAT_NO_IR)
test_feat_nir  = build_feat_dict_nir(X_test_num_nir,  X_test2,  ENG_CAT_NO_IR)

BATCH_NIR = 512
def make_ds_nir(feat_dict, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((feat_dict, labels.values.astype("float32")))
    if shuffle:
        ds = ds.shuffle(10_000, seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_NIR).prefetch(tf.data.AUTOTUNE)

train_ds_nir = make_ds_nir(train_feat_nir, y_train2, shuffle=True)
val_ds_nir   = make_ds_nir(val_feat_nir,   y_val2)
test_ds_nir  = make_ds_nir(test_feat_nir,  y_test2)

# ── Build identical architecture (no IR) ─────────────────────────────────────
tf.keras.backend.clear_session()

num_inp_nir = keras.Input(shape=(len(ENG_NUM_NO_IR),), name="numeric", dtype=tf.float32)
num_branch_nir = normalizer_nir(num_inp_nir)

cat_inps_nir, cat_branches_nir = [], []
for feat in ENG_CAT_NO_IR:
    inp = keras.Input(shape=(), name=feat, dtype=tf.string)
    cat_inps_nir.append(inp)
    idx = string_lookup_nir[feat](inp)
    oh  = layers.CategoryEncoding(
        num_tokens=string_lookup_nir[feat].vocabulary_size(), output_mode="one_hot"
    )(idx)
    cat_branches_nir.append(oh)

all_feats_nir = layers.Concatenate()([num_branch_nir] + cat_branches_nir)
x_nir = layers.Dense(128, activation="relu")(all_feats_nir)
x_nir = layers.Dropout(0.20)(x_nir)
out_nir = layers.Dense(1, activation="sigmoid", dtype="float32")(x_nir)

model_nir = keras.Model(inputs=[num_inp_nir] + cat_inps_nir, outputs=out_nir, name="pre_pricing")
model_nir.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[keras.metrics.AUC(curve="PR", name="pr_auc"), keras.metrics.AUC(curve="ROC", name="roc_auc")],
)

cb_nir = [
    keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=10, restore_best_weights=True, verbose=0),
    keras.callbacks.ReduceLROnPlateau(monitor="val_pr_auc", mode="max", factor=0.5, patience=4, min_lr=1e-6, verbose=0),
    keras.callbacks.TerminateOnNaN(),
]

model_nir.fit(train_ds_nir, validation_data=val_ds_nir, epochs=100, callbacks=cb_nir, verbose=0)

nir_val_proba  = model_nir.predict(val_ds_nir,  verbose=0).ravel()
nir_test_proba = model_nir.predict(test_ds_nir, verbose=0).ravel()

# ── Comparison metrics ────────────────────────────────────────────────────────
def precision_at_k(y_true, proba, k=0.10):
    n = len(y_true)
    n_flagged = max(1, int(np.ceil(k * n)))
    idx = np.argsort(proba)[::-1][:n_flagged]
    return y_true[idx].sum() / n_flagged

def recall_at_k(y_true, proba, k=0.10):
    n = len(y_true)
    n_flagged = max(1, int(np.ceil(k * n)))
    idx = np.argsort(proba)[::-1][:n_flagged]
    return y_true[idx].sum() / y_true.sum()

def lift_at_k(y_true, proba, k=0.10):
    base = y_true.mean()
    return precision_at_k(y_true, proba, k) / base if base > 0 else 0

comparison_rows = []
for model_name, y_true, proba in [
    ("Booked (with InterestRate)", y_test_arr, test_proba),
    ("Pre-pricing (no InterestRate)", y_test_arr, nir_test_proba),
]:
    comparison_rows.append({
        "model"          : model_name,
        "PR-AUC"         : round(average_precision_score(y_true, proba), 4),
        "ROC-AUC"        : round(roc_auc_score(y_true, proba), 4),
        "Precision@10%"  : round(precision_at_k(y_true, proba, 0.10), 4),
        "Recall@10%"     : round(recall_at_k(y_true, proba, 0.10), 4),
        "Lift@10%"       : round(lift_at_k(y_true, proba, 0.10), 4),
    })

df_ir_comparison = pd.DataFrame(comparison_rows)
display(df_ir_comparison)

print("""
InterestRate discussion
-----------------------
InterestRate is typically set AFTER an underwriting decision, reflecting the
borrower's assessed risk. Including it at prediction time can create a proxy
leakage: the model partially recovers the lender's own prior risk estimate
rather than independently predicting default. In a pre-pricing scenario (e.g.,
application scoring before rate setting) the feature is unavailable. The
comparison above quantifies the performance drop when it is excluded.
""")

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9 — FINAL CONCLUSIONS
# ─────────────────────────────────────────────────────────────────────────────

top3_features = df_importance.head(3)["feature"].tolist()
brier = brier_score_loss(y_test_arr, test_proba)
ir_drop_pr_auc = df_ir_comparison.iloc[0]["PR-AUC"] - df_ir_comparison.iloc[1]["PR-AUC"]

conclusion = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                         FINAL CONCLUSIONS                                  ║
╚══════════════════════════════════════════════════════════════════════════════╝

Architecture & Hyperparameters
  • Simple feed-forward network: Dense(128, ReLU) → Dropout(0.20) → Dense(1, Sigmoid)
  • Numeric branch: Keras Normalization layer
  • Categorical branch: StringLookup + CategoryEncoding (one-hot)
  • Optimizer: Adam (lr=1e-3, ReduceLROnPlateau patience=4, factor=0.5)
  • Batch size: {BASELINE_BATCH_SIZE}  |  EarlyStopping patience=10 on val PR-AUC
  • Imbalance strategy: None (raw class proportions); class weights or
    oversampling would be the natural next step.

Training outcome
  • Best epoch        : {best_epoch}
  • Training time     : {round(training_time_sec, 1)} s
  • Val PR-AUC (Keras): {best_val_pr_auc_keras:.4f}
  • Val PR-AUC (AP)   : {baseline_val_pr_auc:.4f}
  • Test PR-AUC (AP)  : {baseline_test_pr_auc:.4f}
  • Fit assessment    : {fit_note}

Precision-Recall trade-off at business threshold ({business_thresh:.3f})
  • See confusion matrix and df_thresh rows above.

Lift@10% (booked model): {lift_at_k(y_test_arr, test_proba, 0.10):.3f}x
Precision@10%           : {precision_at_k(y_test_arr, test_proba, 0.10):.4f}
Recall@10%              : {recall_at_k(y_test_arr, test_proba, 0.10):.4f}

Most important features (permutation importance)
  Top-3: {", ".join(top3_features)}
  → See df_importance and permutation importance chart for full ranking.

Calibration
  • Brier score: {brier:.4f}  (lower is better; 0 = perfect)
  • Calibration curve plotted above. Check for systematic over/under-confidence.

InterestRate sensitivity
  • PR-AUC drop when InterestRate and its derived features are removed:
    Δ PR-AUC ≈ {ir_drop_pr_auc:.4f}
  • If the model is intended for pre-application or pre-pricing scoring,
    InterestRate should be excluded to avoid post-decision leakage.

Known limitations
  Leakage risk   : InterestRate may encode the lender's prior credit view.
  Fairness       : No fairness audit performed; demographic proxies
                   (Age, MaritalStatus, HasDependents) are present.
  Data quality   : LoanID appears unique per row — excluded from features.
                   No temporal ordering used; potential temporal leakage
                   if the dataset spans multiple time periods.
  Imbalance      : Default rate ≈ {y_train2.mean():.3f}; no SMOTE/class weighting applied.

Prototype adequacy
  • The network achieves meaningful PR-AUC above the no-skill baseline and
    shows lift > 1 in the top-k analysis — adequate as a proof-of-concept.
  • Not production-ready: requires fairness testing, temporal validation,
    calibration adjustment, explainability (SHAP), regulatory review, and
    A/B testing before deployment.

Pre-deployment checklist
  ✗ Temporal train/test split validation
  ✗ Fairness / disparate-impact audit
  ✗ SHAP-based explanation per prediction
  ✗ Proper imbalance handling (class weights, SMOTE)
  ✗ Calibration post-processing (Platt / isotonic)
  ✗ Drift monitoring pipeline
  ✗ Business sign-off on decision threshold
"""

print(conclusion)